# euroloto — Exemples d'utilisation

Ce notebook illustre les principales fonctionnalités du package `euroloto` :
initialisation, analyse statistique, prédiction et visualisation.

---

## 0. Import et initialisation

`euroloto.init()` lit `config.yaml` et charge les tirages depuis **`tirage.xlsx`**
(Loto depuis 1976, EuroMillions depuis 2004).

> **Première utilisation ?** Exécutez `euroloto.build_tirage()` dans la section suivante
> pour créer `tirage.xlsx` en téléchargeant les archives FDJ (~30 s, une seule fois).
> Ensuite `euroloto.update_tirage()` ajoute les nouveaux tirages en quelques secondes.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
%matplotlib inline

import euroloto

# Charge tirage.xlsx via config.yaml (chemin relatif au notebook)
euroloto.init('../config.yaml')

---
## 0.1. Gestion des données FDJ

Deux fonctions pour maintenir `tirage.xlsx` à jour automatiquement.

| Fonction | Quand l'utiliser | Durée |
|----------|-----------------|-------|
| `build_tirage()` | **Une seule fois** — crée le fichier depuis zéro | ~30 s |
| `update_tirage()` | **Après chaque tirage** — ajoute uniquement les nouveaux | ~5 s |

Après `update_tirage()`, relancez `euroloto.init()` pour recharger les données en mémoire.

In [ ]:
# ── Première utilisation : décommenter et exécuter UNE SEULE FOIS ──────────
# euroloto.build_tirage('../tirage.xlsx')   # télécharge ~11 ZIP FDJ (~30 s)
# euroloto.init('../config.yaml')           # recharger après la création

# ── Mise à jour courante (après chaque tirage FDJ) ─────────────────────────
# euroloto.update_tirage('../tirage.xlsx')  # ajoute les nouveaux tirages (~5 s)
# euroloto.init('../config.yaml')           # recharger après la mise à jour
# ───────────────────────────────────────────────────────────────────────────
print("Données en mémoire :")
euroloto.info()

---
## 1. Données brutes

In [ ]:
# DataFrame complet — Loto
df_loto = euroloto.draws('loto')
print(f"{len(df_loto)} tirages Loto")
df_loto.tail(5)

In [ ]:
# DataFrame complet — EuroMillions
df_euro = euroloto.draws('euro')
print(f"{len(df_euro)} tirages EuroMillions")
df_euro.tail(5)

---
## 2. Fréquences

In [ ]:
# Top 10 boules les plus sorties au Loto
euroloto.frequency('loto').nlargest(10).rename('apparitions')

In [ ]:
# Graphique — boules principales Loto
# Rouge = chaud (> moyenne +10 %), Bleu = froid, Gris = neutre
euroloto.plot_frequency('loto')

In [ ]:
# Graphique — numéros chance Loto
euroloto.plot_frequency('loto', label='bonus')

In [ ]:
# Graphique — boules EuroMillions
euroloto.plot_frequency('euro')

---
## 3. Numéros chauds / froids

Comparaison de la fréquence des 50 derniers tirages par rapport à la moyenne globale.

In [ ]:
hc = euroloto.hot_cold('loto', n_recent=50)
print("5 numéros les plus CHAUDS :")
print(hc[hc['statut'] == 'chaud'].head(5)[['total_pct', 'recent_pct', 'delta']].to_string())
print("\n5 numéros les plus FROIDS :")
print(hc[hc['statut'] == 'froid'].tail(5)[['total_pct', 'recent_pct', 'delta']].to_string())

In [ ]:
euroloto.plot_hot_cold('loto', n_recent=50)

---
## 4. Co-occurrences

Combien de fois deux numéros sont sortis ensemble dans le même tirage.

In [ ]:
# Top 10 paires les plus co-occurentes
euroloto.top_pairs('loto', n=10)

In [ ]:
# Heatmap — seuil min_cooc=30 pour ne garder que les paires significatives
euroloto.plot_cooccurrence('loto', top_n=20, min_cooc=30)

In [ ]:
# Sans seuil — toutes les paires du top 15
euroloto.plot_cooccurrence('euro', top_n=15)

---
## 5. Tests statistiques d'uniformité

Vérifient si la distribution observée est compatible avec une loi uniforme
(hypothèse attendue si le jeu est équitable).

In [ ]:
import pandas as pd

rows = []
for kind in ('loto', 'euro'):
    chi2 = euroloto.chi2_test(kind)
    ks   = euroloto.ks_test(kind)
    rows.append({
        'jeu': kind,
        'chi2_stat': chi2['statistic'], 'chi2_p': chi2['pvalue'], 'chi2_uniforme': chi2['is_uniform'],
        'ks_stat':   ks['statistic'],   'ks_p':   ks['pvalue'],   'ks_uniforme':   ks['is_uniform'],
    })
pd.DataFrame(rows).set_index('jeu')

---
## 6. Statistiques sur la somme des boules

In [ ]:
print("Loto")
print(euroloto.sum_stats('loto').to_string())
print("\nEuroMillions")
print(euroloto.sum_stats('euro').to_string())

In [ ]:
euroloto.plot_sum('loto')

---
## 7. Distribution pair / impair

In [ ]:
print("Loto — distribution pair/impair par tirage :")
euroloto.even_odd('loto')

---
## 8. Retard (numéros en attente)

Nombre de tirages depuis la dernière apparition de chaque numéro.

In [ ]:
print("Top 10 numéros les plus en retard — Loto :")
euroloto.overdue('loto').head(10)

In [ ]:
euroloto.plot_overdue('euro')

---
## 9. Tendance temporelle

Fréquence normalisée (% par tirage) des 10 numéros les plus fréquents, année par année.
La normalisation corrige le saut dû au passage d'EuroMillions à 2 tirages/semaine en 2011.

In [ ]:
euroloto.plot_temporal('euro', top_n=10)

In [ ]:
euroloto.plot_temporal('loto', top_n=8)

---
## 10. Écarts entre apparitions (gaps)

In [ ]:
euroloto.plot_gaps('loto')

---
## 11. Compléments pour des numéros fixés

**Question :** si je joue toujours les numéros 26 et 27, quels sont les 3 meilleurs compléments
selon l'historique ?

In [ ]:
FIXED = [26, 27]

# Analyse sur le Loto uniquement
print(f"Loto — tirages contenant {FIXED} :")
euroloto.companions(FIXED, kind='loto')

In [ ]:
euroloto.plot_companions(FIXED, kind='loto')

In [ ]:
# Analyse croisée Loto + EuroMillions (plage commune 1–49)
print(f"Loto + EuroMillions — compléments pour {FIXED} :")
euroloto.companions(FIXED, kind='all', n_top=15)

In [ ]:
euroloto.plot_companions(FIXED, kind='all')

In [ ]:
# Exemple avec un seul numéro fixé
euroloto.companions([13], kind='loto', n_top=10)

---
## 12. Prédiction

Génération de combinaisons candidates par Monte Carlo pondéré.

**`alpha`** contrôle l'équilibre entre fréquence et retard :
- `alpha=1.0` → favorise les numéros historiquement fréquents
- `alpha=0.0` → favorise les numéros les plus en retard
- `alpha=0.6` → réglage par défaut (60 % fréquence / 40 % retard)

> **Rappel :** les tirages sont indépendants et équiprobables par construction.
> Ce modèle explore des tendances empiriques — il ne prédit pas l'avenir.

In [ ]:
# 10 combinaisons Loto libres
combos = euroloto.prediction(kind='loto', n=10, alpha=0.6, seed=42)
for i, c in enumerate(combos, 1):
    print(f"#{i:02d}  {c['main']}  +  Chance {c['bonus']}   score={c['score']:.2f}")

In [ ]:
# 10 combinaisons EuroMillions libres
combos_euro = euroloto.prediction(kind='euro', n=10, alpha=0.5, seed=0)
for i, c in enumerate(combos_euro, 1):
    print(f"#{i:02d}  {c['main']}  +  Étoiles {c['bonus']}   score={c['score']:.2f}")

In [ ]:
# 10 combinaisons Loto incluant obligatoirement les numéros [26, 27]
combos_fixed = euroloto.prediction(FIXED, kind='loto', n=10, alpha=0.6, seed=42)
for i, c in enumerate(combos_fixed, 1):
    main_str = '  '.join(f'[{x:02d}]' if x in FIXED else f' {x:02d} ' for x in c['main'])
    print(f"#{i:02d}  {main_str}  +  Chance {c['bonus']}   score={c['score']:.2f}")
print("  ([ ] = numéros fixés)")

In [ ]:
# Prédiction croisée Loto + EuroMillions avec numéros fixés
# Retourne un dict {'loto': [...], 'euro': [...]}
result_all = euroloto.prediction(FIXED, kind='all', n=5, alpha=0.6, seed=42)

print("=== Loto ===")
for i, c in enumerate(result_all['loto'], 1):
    print(f"  #{i}  {c['main']}  +  Chance {c['bonus']}   score={c['score']:.2f}")

print("\n=== EuroMillions ===")
for i, c in enumerate(result_all['euro'], 1):
    print(f"  #{i}  {c['main']}  +  Étoiles {c['bonus']}   score={c['score']:.2f}")

In [ ]:
# Influence du paramètre alpha
print("alpha=1.0 (fréquence pure)")
for c in euroloto.prediction(kind='loto', n=3, alpha=1.0, seed=1):
    print(f"  {c['main']}")

print("\nalpha=0.0 (retard pur)")
for c in euroloto.prediction(kind='loto', n=3, alpha=0.0, seed=1):
    print(f"  {c['main']}")

---
## 13. Combinaisons probables

À partir de deux numéros fixés, génère toutes les combinaisons candidates,
les score par **lift conditionnel moyen** (mesure l'association statistique entre tous
les numéros de la combinaison dans les tirages historiques), puis sélectionne les
*N* meilleures en maximisant la **diversité inter-combinaisons** (distance de Jaccard)
afin d'éviter les quasi-doublons.

| Colonne | Définition |
|---------|-----------|
| `score` | Lift conditionnel moyen sur toutes les C(5,2)=10 paires de la combinaison |
| `prob_pct` | Probabilité relative (%) — score normalisé sur l'ensemble des candidats |

**Référence** (ligne orange pointillée) = `prob_pct` moyen sur *toutes* les
C(`n_candidates`, 3) combinaisons énumérées — le seuil à dépasser pour qu'une
combinaison soit considérée comme significativement chaude.

In [ ]:
# ── Paramètres ──────────────────────────────────────────────────────────────
FIXED_C     = [26, 46]   # numéros fixés
KIND_C      = 'euro'     # 'loto' | 'euro'
N_TOP_C     = 10         # combinaisons à retenir
N_CAND_C    = 35         # compagnons candidats à énumérer (C(N_CAND_C, 3) combos)
# ────────────────────────────────────────────────────────────────────────────

In [ ]:
from IPython.display import display
import pandas as pd

combos_c, ref_c = euroloto.top_combinations(
    FIXED_C, kind=KIND_C, n_top=N_TOP_C, n_candidates=N_CAND_C
)

# Référence en prob_pct : rescale du score moyen vers l'espace prob_pct
n_enum = N_CAND_C  # approximation; on affiche la valeur brute en note
ref_pct_display = ref_c / combos_c['score'].sum() * 100 * N_TOP_C if combos_c['score'].sum() > 0 else 0

print(f"Jeu            : {KIND_C.upper()}")
print(f"Numéros fixés  : {sorted(FIXED_C)}")
print(f"Score référence (lift moyen toutes combos) : {ref_c:.4f}")
print(f"Combinaisons retenues : {len(combos_c)}\n")

# Affichage stylisé
def _fmt_main(lst):
    return '  '.join(f'[{x:02d}]' if x in FIXED_C else f' {x:02d} ' for x in lst)

_display_df = combos_c.copy()
_display_df['combinaison'] = _display_df['main'].apply(_fmt_main)
_display_df = _display_df[['rank', 'combinaison', 'score', 'prob_pct']].copy()
_display_df.columns = ['#', 'Combinaison  ([ ] = fixés)', 'Score lift', 'Prob. rel. (%)']
_display_df = _display_df.set_index('#')

_st = (_display_df.style
       .background_gradient(subset=['Score lift'], cmap='YlOrRd')
       .background_gradient(subset=['Prob. rel. (%)'], cmap='Blues')
       .format({'Score lift': '{:.4f}', 'Prob. rel. (%)': '{:.5f}'}))
display(_st)

In [ ]:
import matplotlib.pyplot as plt

fig_c = euroloto.plot_combinations(FIXED_C, combos_c, ref_c, kind=KIND_C)
display(fig_c)
plt.close(fig_c)

---
## 14. Prédiction complète — workflow depuis une paire de numéros

Pipeline de bout en bout pour construire une grille à partir de deux numéros fixés.

```
FIXED (ex. [26, 46])
     │
     ▼
 Étape 1 ─ Compagnons       : fréquence + lift de chaque numéro associé
     │
     ▼
 Étape 2 ─ Analyse profonde  : cascade de co-occurrences conditionnelles
     │       (matrice 3e → 4e → 5e numéro)
     ▼
 Étape 3 ─ Matrices d'affinité : heatmaps lift conditionnel à chaque étape
     │
     ▼
 Étape 4 ─ Top combinaisons  : N meilleures combos diversifiées (lift + Jaccard)
     │
     ▼
 Étape 5 ─ Grilles finales   : Monte Carlo pondéré avec étoiles / numéro chance
```

> **Rappel :** les tirages sont indépendants et équiprobables par construction.
> Ces analyses explorent des *tendances empiriques* — elles ne prédisent pas l'avenir.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
#  PARAMÈTRES  — seul bloc à modifier
# ════════════════════════════════════════════════════════════════════════════
PRED_FIXED   = [26, 46]   # numéros fixés (2 ou plus)
PRED_KIND    = 'euro'     # 'loto' | 'euro'
PRED_N_TOP   = 10         # nombre de combinaisons à retenir (étapes 4 & 5)
PRED_N_CAND  = 35         # candidats compagnons pour top_combinations
PRED_K_SEEDS = 3          # tirages-graines pour la méthode filtrée (deep_companions)
PRED_ALPHA   = 0.6        # équilibre fréquence / retard  (1.0 = freq. pure)
PRED_METRIC  = 'lift'     # métrique des heatmaps : 'lift' | 'count'
# ════════════════════════════════════════════════════════════════════════════

### Étape 1 — Compagnons (fréquence + lift)

`lift > 1` → le numéro apparaît **plus souvent** avec la paire fixée qu'attendu par le hasard.  
`lift < 1` → association non significative.

In [ ]:
from IPython.display import display
import pandas as pd

comp_p = euroloto.companions(PRED_FIXED, kind=PRED_KIND, n_top=20)
print(f"Compagnons de {PRED_FIXED} ({PRED_KIND}) — top 20 par fréquence\n")

_st_comp = comp_p.style
if 'lift' in comp_p.columns:
    _st_comp = (_st_comp
                .background_gradient(subset=['frequence'], cmap='Blues')
                .background_gradient(subset=['lift'], cmap='RdYlGn', vmin=0.5, vmax=4)
                .format({'pct': '{:.1f}%', 'lift': '{:.2f}'}))
else:
    _st_comp = (_st_comp
                .background_gradient(subset=['total'], cmap='Blues')
                .format({'pct_%': '{:.1f}%'}))
display(_st_comp)
euroloto.plot_companions(PRED_FIXED, kind=PRED_KIND, n_top=15)

### Étape 2 — Analyse profonde (cascade de co-occurrences)

Deux méthodes comparées à chaque étape :
- **GLOBAL** : tous les tirages contenant la paire fixée
- **FILTRÉE** : restreint aux tirages partageant ≥ 1 numéro avec les `k_seeds` tirages-graines les plus récents

La flèche `←` indique quand les deux méthodes divergent.

In [ ]:
deep_result = euroloto.deep_companions(
    PRED_FIXED, kind=PRED_KIND,
    n_top=8, k_seeds=PRED_K_SEEDS,
)

### Étape 3 — Matrices d'affinité (heatmaps)

Chaque heatmap montre le **lift conditionnel** entre les candidats compagnons
dans les tirages contenant l'ensemble fixé courant.
- Cellule orange/rouge = les deux numéros apparaissent ensemble **significativement plus** qu'attendu
- Le meilleur candidat (fréquence max) est sélectionné automatiquement pour passer à l'étape suivante

In [ ]:
import matplotlib.pyplot as plt

figs_deep = euroloto.plot_deep_companions(
    PRED_FIXED, kind=PRED_KIND,
    n_top=12, min_affinity=1, metric=PRED_METRIC,
)
for fig in figs_deep:
    display(fig)
    plt.close(fig)

### Étape 4 — Top combinaisons (lift × diversité)

Toutes les C(`n_candidates`, 3) combinaisons possibles sont scorées par **lift conditionnel moyen**,
puis les `n_top` meilleures sont sélectionnées en maximisant la **distance de Jaccard**
entre elles (évite les quasi-doublons).

La ligne orange = score moyen sur **tous** les candidats énumérés (référence à battre).

In [ ]:
top_c, ref_score = euroloto.top_combinations(
    PRED_FIXED, kind=PRED_KIND,
    n_top=PRED_N_TOP, n_candidates=PRED_N_CAND,
)

print(f"Score reference (moy. tous candidats) : {ref_score:.4f}\n")

# Tableau stylise
def _highlight_fixed(lst, fixed=PRED_FIXED):
    return '  '.join(f'[{x:02d}]' if x in fixed else f' {x:02d} ' for x in lst)

_tc_disp = top_c.copy()
_tc_disp['combinaison'] = _tc_disp['main'].apply(_highlight_fixed)
_tc_disp = _tc_disp[['rank', 'combinaison', 'score', 'prob_pct']].set_index('rank')
_tc_disp.columns = ['Combinaison  ([ ]=fixes)', 'Score lift', 'Prob. rel. (%)']

display(_tc_disp.style
        .background_gradient(subset=['Score lift'],     cmap='YlOrRd')
        .background_gradient(subset=['Prob. rel. (%)'], cmap='Blues')
        .format({'Score lift': '{:.4f}', 'Prob. rel. (%)': '{:.5f}'}))

fig_hist = euroloto.plot_combinations(PRED_FIXED, top_c, ref_score, kind=PRED_KIND)
display(fig_hist)
plt.close(fig_hist)

### Étape 5 — Grilles finales (Monte Carlo pondéré)

Les combinaisons de l'étape 4 fournissent un **sous-espace de numéros chauds**.
Le Monte Carlo pondéré (`prediction`) tire aléatoirement `n` grilles complètes
en respectant les contraintes :
- Les numéros `PRED_FIXED` sont **toujours inclus**
- Les probabilités de tirage sont proportionnelles à `freq^alpha × retard^(1-alpha)`
- Chaque grille inclut son **bonus** (étoile EuroMillions ou numéro chance Loto)

Le paramètre `alpha` (`PRED_ALPHA`) règle l'équilibre :
| `alpha` | Favorise |
|---------|---------|
| `1.0`   | numéros historiquement fréquents |
| `0.5`   | compromis fréquence / retard |
| `0.0`   | numéros les plus en retard |

In [ ]:
import pandas as pd

# ── Monte Carlo pondéré — grilles finales ────────────────────────────────────
grilles = euroloto.prediction(
    PRED_FIXED,
    kind=PRED_KIND,
    n=PRED_N_TOP,
    alpha=PRED_ALPHA,
    seed=42,
)

# ── Affichage formaté ────────────────────────────────────────────────────────
bonus_label = 'Etoiles' if PRED_KIND == 'euro' else 'Chance'

rows = []
for i, g in enumerate(grilles, 1):
    main_str = '  '.join(
        f'[{x:02d}]' if x in PRED_FIXED else f' {x:02d} '
        for x in sorted(g['main'])
    )
    bonus_str = '  '.join(str(b) for b in g['bonus'])
    rows.append({
        '#': i,
        f'Combinaison  ([ ]=fixes, {PRED_KIND})': main_str,
        bonus_label: bonus_str,
        'Score': round(g['score'], 3),
    })

_df_grilles = pd.DataFrame(rows).set_index('#')
print(f"=== {PRED_N_TOP} grilles finales | {PRED_KIND.upper()} | alpha={PRED_ALPHA} ===\n")

display(
    _df_grilles.style
    .background_gradient(subset=['Score'], cmap='YlOrRd')
    .format({'Score': '{:.3f}'})
)

print("\n([ ] = numero(s) fixe(s)  |  score = Monte Carlo pondéré freq/retard)")